In [1]:
import transformers
import torch
import psutil

/home/eauten2s/.local/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
torch.cuda.is_available()

False

In [3]:
print(f"Anzahl der Kerne: {psutil.cpu_count(logical=True)}")
print(f"Total: {psutil.virtual_memory().total / (1024 ** 3):.2f} GB")

Anzahl der Kerne: 128
Total: 377.10 GB


In [4]:
if torch.cuda.is_available():
    print("CUDA ist verfügbar. Folgende GPUs sind erreichbar:")
    num_gpus = torch.cuda.device_count()
    print(f"Anzahl verfügbarer GPUs: {num_gpus}")
    for i in range(num_gpus):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

In [5]:
model_id =  "meta-llama/Meta-Llama-3-8B-Instruct" # "meta-llama/Meta-Llama-3-8B" #

pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device="cpu",# cuda
)

2024-04-23 14:02:13.778928: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-04-23 14:02:13.819351: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-04-23 14:02:14.433116: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/usr/lib64/python3.9/site-packages/scipy/__init__.py:138: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.0)
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxve

In [6]:
prompt = """
Convert the following medical trial eligibility criteria into a structured JSON format:
Inclusion Criteria:
- Overweight or obese subjects [according to body mass index (BMI)]
- Fasting plasma glucose value between 100 and 125 mg/dl, with impaired fasting glucose or impaired glucose tolerance confirmed with oral glucose tolerance test (OGTT)
- Total cholesterol values ≥ 200 mg/dl

Exclusion Criteria:
- Patients with neoplastic and liver diseases, renal failure
- Patients with type 1 or 2 diabetes mellitus
- Pregnant or breastfeeding women
- Hypersensitivity to any of the ingredients
- Therapy with lipid-lowering drugs
- Use of products containing red yeast rice
"""

template = "{\"InclusionCriteria\":{\"IC1\":\"...\",\"IC2\":\"...\",\"ICx\":\"...\"},\"ExclusionCriteria\":{\"EC1\":\"...\",\"EC2\":\"...\",\"EC3\":\"...\",\"ECx\":\"...\",}}"
template_logic = "{\"AND\":{\"IC1\":{\"IC1\":\"...\",\"OR\":{\"IC1.x\":\"...\",}},\"IC2\":{\"IC2\":\"...\",\"AND\":{\"IC2.1\":\"...\",\"OR\":{\"IC2.x\":\"...\",}}},\"IC3\":\"...\"},\"NOTOR\":{\"EC1\":{\"EC1\":\"...\",\"OR\":{\"EC1.1\":\"...\",\"EC1.2\":\"...\",\"EC1.3\":\"...\"}},\"EC2\":{\"EC2\":\"...\",\"OR\":{\"EC2.1\":\"...\",\"EC2.2\":\"...\"}},\"EC3\":{\"EC3\":\"...\",\"OR\":{\"EC3.1\":\"...\",\"EC3.2\":\"...\"}},\"EC4\":\"...\",\"ECx\":\"...\"}}"
messages = [
    {"role": "system", "content": f"You are a Elegibility Criteria to JSON machine. Return inclusion and exclusion criteria as JSON object. For every critera a key and the description as values. Take only the descriptions, add nothing extra, and number them.If possible, further subdivide the criteria logically as in the template.. Use this template: {template_logic}"},
    {"role": "user", "content": f"{prompt}"},
]

prompt = pipeline.tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
)

terminators = [
    pipeline.tokenizer.eos_token_id,
    pipeline.tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

In [7]:
outputs = pipeline(
    prompt,
    max_new_tokens=500,
    eos_token_id=terminators,
    do_sample=True,
    temperature=0.6,# deterministich - kreativ
    top_p=0.9,
)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [ ]:
print(outputs[0]["generated_text"][len(prompt):])

Here is the eligibility criteria in a structured JSON format:

{
  "AND": {
    "IC1": {
      "IC1": "Overweight or obese subjects [according to body mass index (BMI)]"
    },
    "IC2": {
      "IC2": "Fasting plasma glucose value between 100 and 125 mg/dl, with impaired fasting glucose or impaired glucose tolerance confirmed with oral glucose tolerance test (OGTT)"
    },
    "IC3": {
      "IC3": "Total cholesterol values ≥ 200 mg/dl"
    }
  }
}

{
  "NOTOR": {
    "EC1": {
      "EC1": "Patients with neoplastic and liver diseases, renal failure"
    },
    "EC2": {
      "EC2": "Patients with type 1 or 2 diabetes mellitus"
    },
    "EC3": {
      "EC3": "Pregnant or breastfeeding women"
    },
    "EC4": {
      "EC4": "Hypersensitivity to any of the ingredients"
    },
    "EC5": {
      "EC5": "Therapy with lipid-lowering drugs"
    },
    "EC6": {
      "EC6": "Use of products containing red yeast rice"
    }
  }
}
